### Dictionary

In [6]:
import pandas as pd

df_dicionario = pd.read_csv('dictionary.csv')

pd.set_option('display.max_colwidth', None)
df_styled = df_dicionario.style.set_properties(**{'text-align': 'left'})

display(df_dicionario)

,Name,Description (Semantic Meaning),Data Type,Units,Possible Values / Categories,Potential Data Quality Issues
0,product_id,Product ID,Text,-,Alphanumeric strings,Duplicated products
1,product_name,Name of the Product,Text,-,Text,"Long strings, special characters, too much information"
2,category,Category of the Product,Categorical,-,"e.g., Electronics, Computers, Home&Kitchen",Values are grouped using a pipe |
3,discounted_price,Discounted Price of the Product,Numerical,Currency (₹),> 0,Contains currency symbol and commas. Must be cleaned and converted to float.
4,actual_price,Actual Price of the Product,Numerical,Currency (₹),> 0,Contains currency symbol and commas. Must be cleaned and converted to float.
5,discount_percentage,Percentage of Discount for the Product,Numerical,Percentage (%),0% to 100%,Contains '%' symbol.
6,rating,Rating of the Product,Numerical,Stars,1.0 to 5.0,Some rows contain invalid characters
7,rating_count,Number of people who voted for the Amazon rating,Numerical,Count,> 0,Contains commas to separate thousands
8,about_product,Description about the Product,Text,-,Text,Contains special characters and diferent formats of text
9,user_id,ID of the user who wrote review for the Product,Text,-,Alphanumeric strings,Multiple IDs are concatenated.


# Load amazon.csv

In [7]:
import pandas as pd

df = pd.read_csv('amazon.csv')
print(f"Dataset shape before cleaning: {df.shape}")

Dataset shape before cleaning: (1465, 16)


### Duplicate Detection and Handling

**Detection Strategy:**
To identify duplicates, the `product_id` column was used. The `product_id` is the primary key of the product. The same product can be collected more than once with some variation in value in another column, but the `product_id` remains the same. That is, any repeated `product_id` is a duplicate.

**Consolidation Method:**
After identifying the duplicates based on the `product_id`, the chosen consolidation method was removal. I kept the first occurrence among the duplicates. This approach is justified because if we keep the duplicate products, it will alter our future results.

In [17]:
# Identifying exact row duplicates
exact_duplicates = df.duplicated().sum()
print(f"Exact row duplicates: {exact_duplicates}")

# Identifying duplicates by product_id
product_id_duplicates = df.duplicated(subset=['product_id']).sum()
print(f"Duplicates by product_id: {product_id_duplicates}")

# Applying the removal (keeping the first record)
df_cleaned = df.drop_duplicates(subset=['product_id'], keep='first')

print(f"Dataset after removing duplicates: {df_cleaned.shape}")

df_cleaned.to_csv('amazon_after_duplicate_removal.csv', index=False)

Exact row duplicates: 0
Duplicates by product_id: 114
Dataset after removing duplicates: (1351, 16)


### Missing Data Treatment

In [18]:
# Missing values per variable
missing_per_variable = df_cleaned.isnull().sum()
print("Missing values per variable:")
print(missing_per_variable[missing_per_variable > 0])

# Missing values overall
total_missing = df_cleaned.isnull().sum().sum()
print(f"\nTotal missing values in the dataset: {total_missing}")

Missing values per variable:
rating_count    2
dtype: int64

Total missing values in the dataset: 2


**Data Missing Patterns:**
I found missing values in the `rating_count` column. This pattern can be classified as MNAR. When a product has no ratings, it is likely that it has zero ratings.

**Treatment Strategy and Justification:**
Removal of cases with missing data.

**Justification:** Since only 2 of the more than 1,000 records contain missing data, applying the exclusion of cases with missing data is the most efficient and safe approach. It eliminates the data quality problem without causing information loss.

In [21]:
df_cleaned = df_cleaned.dropna(subset=['rating_count'])

print(f"Total missing values after treatment: {df_cleaned.isnull().sum().sum()}")

print(f"\nDataset after removing missing data: {df_cleaned.shape}")

df_cleaned.to_csv('amazon_after_missing_data_treatment.csv', index=False)

Total missing values after treatment: 0

Dataset after removing missing data: (1349, 16)
